In [4]:
# Importing Libraries
import pandas as pd
import warnings

from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem import Descriptors
from rdkit.Chem import AllChem, PandasTools
from rdkit.ML.Descriptors import MoleculeDescriptors
import pubchempy as pcp

import requests
from tqdm import tqdm

In [46]:
# The bcf_data.xlsx file contains the CAS numbers and their corresponding Dragon descriptors. This dataset can be obtained from Grisoni, F., Consonni, V., Vighi, M., Villa, S., Todeschini, R., 2016. Investigating the mechanisms of bioconcentration through QSAR classification trees. Environ Int 88, 198-205. https://doi.org/10.1016/j.envint.2015.12.024.
df = pd.read_excel('bcf_data.xlsx')
df.to_csv('bcf_data_original.csv', index=False)

In [47]:
df = pd.read_csv('bcf_data_original.csv')

In [48]:
import pandas as pd

# 16 CAS numbers conatining multiple entries separated by '/' or ',' that were not able to be converted to SMILES using PubChemPy/Cactus API
cas_numbers = [
    '127-20-8,CAE_NA_01', '133855-98-8/106325-08-0', '1461-22-9,56573-85-4',
    '1490-04-6/2216-51-5', '25154-54-5', '294-62-2,830-13-7',
    '35822-46-9,37871-00-4', '39515-41-8/64257-84-7', '504-03-0,766-17-6',
    '55179-31-2/70585-36-3', '57837-19-1,70630-17-0', '60-57-1/72-20-8',
    '66441-23-4,71283-80-2', '69806-40-2/72619-32-0', '91-17-8/493-01-6/493-02-7',
    '95-47-6,1330-20-7'
]


In [49]:
# 16 CAS numbers conatining multiple entries separated by '/' or ',' that were not able to be converted to SMILES using PubChemPy/Cactus API
filtered_df = df[df['CAS'].isin(cas_numbers)][['CAS']]
filtered_df

,CAS
143,"127-20-8,CAE_NA_01"
163,133855-98-8/106325-08-0
184,"1461-22-9,56573-85-4"
187,1490-04-6/2216-51-5
286,25154-54-5
311,"294-62-2,830-13-7"
365,"35822-46-9,37871-00-4"
387,39515-41-8/64257-84-7
416,"504-03-0,766-17-6"
451,55179-31-2/70585-36-3


In [ ]:
# Generate QSAR-ready SMILES without stereochemistry
def qsar_ready_smiles(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol:
            return Chem.MolToSmiles(mol, canonical=True, isomericSmiles=False)
        else:
            return None
    except:
        return None

# Generate SMILES for each CAS RN using PubChem, CACTUS API, or RDKit with a progress bar
def get_smiles_from_cas(cas_number):
    try:
        # First try using the PubChem API
        url_pubchem = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{cas_number}/property/IsomericSMILES/JSON"
        response = requests.get(url_pubchem)
        if response.status_code == 200:
            data = response.json()
            return data['PropertyTable']['Properties'][0]['IsomericSMILES']
    except (requests.exceptions.RequestException, KeyError, IndexError):
        pass
    
    try:
        # If PubChem fails, use the CACTUS API
        url_cactus = f"https://cactus.nci.nih.gov/chemical/structure/{cas_number}/smiles"
        response = requests.get(url_cactus)
        response.raise_for_status()
        return response.text.strip()
    except requests.exceptions.RequestException:
        pass
    
    try:
        # If all else fails, use RDKit to generate a placeholder structure
        mol = Chem.MolFromSmiles(cas_number) 
        if mol:
            return Chem.MolToSmiles(mol)
    except Exception as e:
        pass
    
    return None

In [ ]:
# Generate SMILES for the entire dataset using PubChem, CACTUS API, or RDKit with a progress bar
# The CAS with different entries separated by '/' or ',' were not able to be converted to SMILES using PubChemPy/Cactus API, so they will be skipped.
tqdm.pandas(desc="Generating SMILES")
df.loc[:, 'SMILES'] = df['CAS'].progress_apply(get_smiles_from_cas)

Generating SMILES:  18%|█▊        | 144/779 [00:36<02:19,  4.57it/s][14:53:50] SMILES Parse Error: syntax error while parsing: 127-20-8,CAE_NA_01
[14:53:50] SMILES Parse Error: check for mistakes around position 1:
[14:53:50] 127-20-8,CAE_NA_01
[14:53:50] ^
[14:53:50] SMILES Parse Error: Failed parsing SMILES '127-20-8,CAE_NA_01' for input: '127-20-8,CAE_NA_01'
Generating SMILES:  21%|██        | 164/779 [00:41<02:32,  4.04it/s][14:53:55] SMILES Parse Error: syntax error while parsing: 133855-98-8/106325-08-0
[14:53:55] SMILES Parse Error: check for mistakes around position 1:
[14:53:55] 133855-98-8/106325-08-0
[14:53:55] ^
[14:53:55] SMILES Parse Error: Failed parsing SMILES '133855-98-8/106325-08-0' for input: '133855-98-8/106325-08-0'
Generating SMILES:  24%|██▎       | 185/779 [00:48<03:15,  3.03it/s][14:54:02] SMILES Parse Error: syntax error while parsing: 1461-22-9,56573-85-4
[14:54:02] SMILES Parse Error: check for mistakes around position 1:
[14:54:02] 1461-22-9,56573-85-4
[14

In [ ]:
# Check for NaN values in the dataframe
nan_count = df.isna().sum()

# Display the result
nan_count

CAS        0
SMILES    16
dtype: int64

In [ ]:
# Remove rows with any NaN values
df = df.dropna()

In [60]:
# Drop the specified columns from the DataFrame
columns_to_remove = [ "Set", "logBCF", "logKOW", "nHM", "piPC09", "PCD", "X2Av", "MLOGP", "ON1V", "N-072", "B02[C-N]", "F04[C-O]"]
df = df.drop(columns=columns_to_remove)

In [61]:
# Check for NaN values in the dataframe
nan_count = df.isna().sum()

# Display the result
nan_count

CAS       0
SMILES    0
Class     0
dtype: int64

In [62]:
df

,CAS,SMILES,Class
0,100-02-7,O=[N+](c1ccc(cc1)O)[O-],1
1,100-17-4,O=[N+](c1ccc(cc1)OC)[O-],1
2,100-18-5,c1cc(ccc1C(C)C)C(C)C,3
3,100-25-4,O=[N+]([O-])c1ccc(cc1)[N+](=O)[O-],3
4,100-40-3,C=CC1CCC=CC1,1
...,...,...,...
758,99-30-9,O=[N+]([O-])c1cc(c(N)c(c1)Cl)Cl,1
759,99387-89-0,FC(F)(F)c2cc(ccc2(N=C(n1cncc1)COCCC))Cl,2
760,99-65-0,O=[N+]([O-])c1cccc(c1)[N+](=O)[O-],1
761,99-71-8,CC(c1ccc(cc1)O)CC,3


In [63]:
# Check for duplicate rows
duplicate_rows = df.duplicated().sum()
print(f"Total duplicate rows: {duplicate_rows}")

Total duplicate rows: 0


In [64]:
# Generate QSAR-ready SMILES without stereochemistry
def qsar_ready_smiles(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol:
            return Chem.MolToSmiles(mol, canonical=True, isomericSmiles=False)
        else:
            return None
    except:
        return None

In [65]:
tqdm.pandas(desc="Generating SMILES")
df.loc[:, 'QSAR_READY_SMILES'] = df['SMILES'].progress_apply(qsar_ready_smiles)

Generating SMILES: 100%|██████████| 763/763 [00:00<00:00, 6284.61it/s]


In [66]:
df

,CAS,SMILES,Class,QSAR_READY_SMILES
0,100-02-7,O=[N+](c1ccc(cc1)O)[O-],1,O=[N+]([O-])c1ccc(O)cc1
1,100-17-4,O=[N+](c1ccc(cc1)OC)[O-],1,COc1ccc([N+](=O)[O-])cc1
2,100-18-5,c1cc(ccc1C(C)C)C(C)C,3,CC(C)c1ccc(C(C)C)cc1
3,100-25-4,O=[N+]([O-])c1ccc(cc1)[N+](=O)[O-],3,O=[N+]([O-])c1ccc([N+](=O)[O-])cc1
4,100-40-3,C=CC1CCC=CC1,1,C=CC1CC=CCC1
...,...,...,...,...
758,99-30-9,O=[N+]([O-])c1cc(c(N)c(c1)Cl)Cl,1,Nc1c(Cl)cc([N+](=O)[O-])cc1Cl
759,99387-89-0,FC(F)(F)c2cc(ccc2(N=C(n1cncc1)COCCC))Cl,2,CCCOCC(=Nc1ccc(Cl)cc1C(F)(F)F)n1ccnc1
760,99-65-0,O=[N+]([O-])c1cccc(c1)[N+](=O)[O-],1,O=[N+]([O-])c1cccc([N+](=O)[O-])c1
761,99-71-8,CC(c1ccc(cc1)O)CC,3,CCC(C)c1ccc(O)cc1


In [67]:
# Check for NaN values in the dataframe
nan_count = df.isna().sum()

# Display the result
nan_count

CAS                  0
SMILES               0
Class                0
QSAR_READY_SMILES    0
dtype: int64

In [68]:
# Remove the 'SMILES' column
df.drop(columns=['SMILES'], inplace=True)

In [69]:
df

,CAS,Class,QSAR_READY_SMILES
0,100-02-7,1,O=[N+]([O-])c1ccc(O)cc1
1,100-17-4,1,COc1ccc([N+](=O)[O-])cc1
2,100-18-5,3,CC(C)c1ccc(C(C)C)cc1
3,100-25-4,3,O=[N+]([O-])c1ccc([N+](=O)[O-])cc1
4,100-40-3,1,C=CC1CC=CCC1
...,...,...,...
758,99-30-9,1,Nc1c(Cl)cc([N+](=O)[O-])cc1Cl
759,99387-89-0,2,CCCOCC(=Nc1ccc(Cl)cc1C(F)(F)F)n1ccnc1
760,99-65-0,1,O=[N+]([O-])c1cccc([N+](=O)[O-])c1
761,99-71-8,3,CCC(C)c1ccc(O)cc1


In [70]:
# Count the number of SMILES that couldn't be converted
failed_QSAR_conversion_count = df['QSAR_READY_SMILES'].isna().sum()
print(f"Number of SMILES that couldn't be converted: {failed_QSAR_conversion_count}")

Number of SMILES that couldn't be converted: 0


In [71]:
# Check for duplicate rows
duplicate_rows = df.duplicated().sum()
print(f"Total duplicate rows: {duplicate_rows}")

Total duplicate rows: 0


In [ ]:
# Save the unique dataframe to a new CSV file
#df.to_csv('SMILES_old.csv', index=False)

In [1]:
import pandas as pd
df = pd.read_csv('SMILES_old.csv')

In [2]:
df

,CAS,Class,QSAR_READY_SMILES
0,100-02-7,1,O=[N+]([O-])c1ccc(O)cc1
1,100-17-4,1,COc1ccc([N+](=O)[O-])cc1
2,100-18-5,3,CC(C)c1ccc(C(C)C)cc1
3,100-25-4,3,O=[N+]([O-])c1ccc([N+](=O)[O-])cc1
4,100-40-3,1,C=CC1CC=CCC1
...,...,...,...
758,99-30-9,1,Nc1c(Cl)cc([N+](=O)[O-])cc1Cl
759,99387-89-0,2,CCCOCC(=Nc1ccc(Cl)cc1C(F)(F)F)n1ccnc1
760,99-65-0,1,O=[N+]([O-])c1cccc([N+](=O)[O-])c1
761,99-71-8,3,CCC(C)c1ccc(O)cc1


In [5]:
# Convert SMILES to RDKit molecules
mol_list = []

for smile in df['QSAR_READY_SMILES']:
  mol = Chem.MolFromSmiles(smile)
  mol = Chem.AddHs(mol)
  mol_list.append(mol)

df = pd.concat([df, pd.DataFrame(mol_list, columns = (['mol']))], axis=1)

In [6]:
df

,CAS,Class,QSAR_READY_SMILES,mol
0,100-02-7,1,O=[N+]([O-])c1ccc(O)cc1,<rdkit.Chem.rdchem.Mol object at 0x000001F1F67...
1,100-17-4,1,COc1ccc([N+](=O)[O-])cc1,<rdkit.Chem.rdchem.Mol object at 0x000001F1F67...
2,100-18-5,3,CC(C)c1ccc(C(C)C)cc1,<rdkit.Chem.rdchem.Mol object at 0x000001F1F67...
3,100-25-4,3,O=[N+]([O-])c1ccc([N+](=O)[O-])cc1,<rdkit.Chem.rdchem.Mol object at 0x000001F1F67...
4,100-40-3,1,C=CC1CC=CCC1,<rdkit.Chem.rdchem.Mol object at 0x000001F1F67...
...,...,...,...,...
758,99-30-9,1,Nc1c(Cl)cc([N+](=O)[O-])cc1Cl,<rdkit.Chem.rdchem.Mol object at 0x000001F1F67...
759,99387-89-0,2,CCCOCC(=Nc1ccc(Cl)cc1C(F)(F)F)n1ccnc1,<rdkit.Chem.rdchem.Mol object at 0x000001F1F67...
760,99-65-0,1,O=[N+]([O-])c1cccc([N+](=O)[O-])c1,<rdkit.Chem.rdchem.Mol object at 0x000001F1F67...
761,99-71-8,3,CCC(C)c1ccc(O)cc1,<rdkit.Chem.rdchem.Mol object at 0x000001F1F67...


## **Generating Molecular Descriptors Using RDKit**

In [7]:
# create another instance for calculating molecular descriptors

Des_func = MoleculeDescriptors.MolecularDescriptorCalculator(x[0] for x in Descriptors._descList)

In [8]:
des = []

for mol in df['mol']:
  des.append(Des_func.CalcDescriptors(mol))

In [9]:
Final_df = pd.concat([df, pd.DataFrame(des, columns=(x[0] for x in Descriptors._descList))], axis=1)


In [10]:
Final_df

,CAS,Class,QSAR_READY_SMILES,mol,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
0,100-02-7,1,O=[N+]([O-])c1ccc(O)cc1,<rdkit.Chem.rdchem.Mol object at 0x000001F1F67...,10.457864,10.457864,0.629074,-1.005484,0.470728,15.800000,...,0,0,0,0,0,0,0,0,0,0
1,100-17-4,1,COc1ccc([N+](=O)[O-])cc1,<rdkit.Chem.rdchem.Mol object at 0x000001F1F67...,10.590532,10.590532,0.837407,-2.971614,0.478623,19.454545,...,0,0,0,0,0,0,0,0,0,0
2,100-18-5,3,CC(C)c1ccc(C(C)C)cc1,<rdkit.Chem.rdchem.Mol object at 0x000001F1F67...,8.110848,8.110848,1.375293,-3.670284,0.617703,39.000000,...,0,0,0,0,0,0,0,0,0,0
3,100-25-4,3,O=[N+]([O-])c1ccc([N+](=O)[O-])cc1,<rdkit.Chem.rdchem.Mol object at 0x000001F1F67...,10.457226,10.457226,1.029907,-1.106674,0.494119,14.000000,...,0,0,0,0,0,0,0,0,0,0
4,100-40-3,1,C=CC1CC=CCC1,<rdkit.Chem.rdchem.Mol object at 0x000001F1F67...,7.925579,7.925579,1.250291,-3.407176,0.452662,79.500000,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
758,99-30-9,1,Nc1c(Cl)cc([N+](=O)[O-])cc1Cl,<rdkit.Chem.rdchem.Mol object at 0x000001F1F67...,10.569109,10.569109,0.004437,-0.953930,0.436427,14.166667,...,0,0,0,0,0,0,0,0,0,0
759,99387-89-0,2,CCCOCC(=Nc1ccc(Cl)cc1C(F)(F)F)n1ccnc1,<rdkit.Chem.rdchem.Mol object at 0x000001F1F67...,13.858852,13.858852,0.050830,-5.501763,0.452489,24.521739,...,0,0,0,0,0,0,0,0,0,0
760,99-65-0,1,O=[N+]([O-])c1cccc([N+](=O)[O-])c1,<rdkit.Chem.rdchem.Mol object at 0x000001F1F67...,10.479335,10.479335,0.935741,-1.143704,0.494119,14.000000,...,0,0,0,0,0,0,0,0,0,0
761,99-71-8,3,CCC(C)c1ccc(O)cc1,<rdkit.Chem.rdchem.Mol object at 0x000001F1F67...,8.182117,8.182117,0.860105,-3.750924,0.686860,37.454545,...,0,0,0,0,0,0,0,0,0,0


In [11]:
# Identify columns with NaN values
nan_columns = Final_df.columns[Final_df.isna().any()].tolist()

# Display columns that contain NaN values
nan_columns


['MaxPartialCharge',
 'MinPartialCharge',
 'MaxAbsPartialCharge',
 'MinAbsPartialCharge',
 'BCUT2D_MWHI',
 'BCUT2D_MWLOW',
 'BCUT2D_CHGHI',
 'BCUT2D_CHGLO',
 'BCUT2D_LOGPHI',
 'BCUT2D_LOGPLOW',
 'BCUT2D_MRHI',
 'BCUT2D_MRLOW']

In [12]:
missing_counts_df = (
    Final_df[nan_columns]
    .isna()
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

missing_counts_df.columns = ["Feature", "Missing_Row_Count"]

missing_counts_df

,Feature,Missing_Row_Count
0,BCUT2D_CHGLO,7
1,BCUT2D_CHGHI,7
2,BCUT2D_MWLOW,7
3,BCUT2D_MWHI,7
4,BCUT2D_LOGPHI,7
5,BCUT2D_LOGPLOW,7
6,BCUT2D_MRHI,7
7,BCUT2D_MRLOW,7
8,MinAbsPartialCharge,5
9,MaxAbsPartialCharge,5


In [14]:
# Identify columns with a single unique value
single_value_columns = [col for col in Final_df.columns if Final_df[col].nunique() == 1]
df_single_value = Final_df[single_value_columns]
df_single_value

,SMR_VSA8,SlogP_VSA9,NumSpiroAtoms,fr_HOCCN,fr_azide,fr_azo,fr_barbitur,fr_benzodiazepine,fr_diazo,fr_dihydropyridine,fr_epoxide,fr_isocyan,fr_isothiocyan,fr_lactam,fr_nitro_arom_nonortho,fr_prisulfonamd,fr_quatN,fr_tetrazole,fr_unbrch_alkane
0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
758,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
759,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
760,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
761,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [15]:
# Drop the single value columns
df_cleaned = Final_df.drop(columns=single_value_columns)
df_cleaned

,CAS,Class,QSAR_READY_SMILES,mol,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,...,fr_priamide,fr_pyridine,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_thiazole,fr_thiocyan,fr_thiophene,fr_urea
0,100-02-7,1,O=[N+]([O-])c1ccc(O)cc1,<rdkit.Chem.rdchem.Mol object at 0x000001F1F67...,10.457864,10.457864,0.629074,-1.005484,0.470728,15.800000,...,0,0,0,0,0,0,0,0,0,0
1,100-17-4,1,COc1ccc([N+](=O)[O-])cc1,<rdkit.Chem.rdchem.Mol object at 0x000001F1F67...,10.590532,10.590532,0.837407,-2.971614,0.478623,19.454545,...,0,0,0,0,0,0,0,0,0,0
2,100-18-5,3,CC(C)c1ccc(C(C)C)cc1,<rdkit.Chem.rdchem.Mol object at 0x000001F1F67...,8.110848,8.110848,1.375293,-3.670284,0.617703,39.000000,...,0,0,0,0,0,0,0,0,0,0
3,100-25-4,3,O=[N+]([O-])c1ccc([N+](=O)[O-])cc1,<rdkit.Chem.rdchem.Mol object at 0x000001F1F67...,10.457226,10.457226,1.029907,-1.106674,0.494119,14.000000,...,0,0,0,0,0,0,0,0,0,0
4,100-40-3,1,C=CC1CC=CCC1,<rdkit.Chem.rdchem.Mol object at 0x000001F1F67...,7.925579,7.925579,1.250291,-3.407176,0.452662,79.500000,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
758,99-30-9,1,Nc1c(Cl)cc([N+](=O)[O-])cc1Cl,<rdkit.Chem.rdchem.Mol object at 0x000001F1F67...,10.569109,10.569109,0.004437,-0.953930,0.436427,14.166667,...,0,0,0,0,0,0,0,0,0,0
759,99387-89-0,2,CCCOCC(=Nc1ccc(Cl)cc1C(F)(F)F)n1ccnc1,<rdkit.Chem.rdchem.Mol object at 0x000001F1F67...,13.858852,13.858852,0.050830,-5.501763,0.452489,24.521739,...,0,0,0,0,0,0,0,0,0,0
760,99-65-0,1,O=[N+]([O-])c1cccc([N+](=O)[O-])c1,<rdkit.Chem.rdchem.Mol object at 0x000001F1F67...,10.479335,10.479335,0.935741,-1.143704,0.494119,14.000000,...,0,0,0,0,0,0,0,0,0,0
761,99-71-8,3,CCC(C)c1ccc(O)cc1,<rdkit.Chem.rdchem.Mol object at 0x000001F1F67...,8.182117,8.182117,0.860105,-3.750924,0.686860,37.454545,...,0,0,0,0,0,0,0,0,0,0


In [18]:
df_cleaned.to_csv('rdkit_data_12_missing_features.csv', index=False)

In [1]:
import pandas as pd

df = pd.read_csv('rdkit_data_12_missing_features.csv')
df

,CAS,Class,QSAR_READY_SMILES,mol,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,...,fr_priamide,fr_pyridine,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_thiazole,fr_thiocyan,fr_thiophene,fr_urea
0,100-02-7,1,O=[N+]([O-])c1ccc(O)cc1,<rdkit.Chem.rdchem.Mol object at 0x000001F1F67...,10.457864,10.457864,0.629074,-1.005484,0.470728,15.800000,...,0,0,0,0,0,0,0,0,0,0
1,100-17-4,1,COc1ccc([N+](=O)[O-])cc1,<rdkit.Chem.rdchem.Mol object at 0x000001F1F67...,10.590532,10.590532,0.837407,-2.971614,0.478623,19.454545,...,0,0,0,0,0,0,0,0,0,0
2,100-18-5,3,CC(C)c1ccc(C(C)C)cc1,<rdkit.Chem.rdchem.Mol object at 0x000001F1F67...,8.110848,8.110848,1.375293,-3.670284,0.617703,39.000000,...,0,0,0,0,0,0,0,0,0,0
3,100-25-4,3,O=[N+]([O-])c1ccc([N+](=O)[O-])cc1,<rdkit.Chem.rdchem.Mol object at 0x000001F1F67...,10.457226,10.457226,1.029907,-1.106674,0.494119,14.000000,...,0,0,0,0,0,0,0,0,0,0
4,100-40-3,1,C=CC1CC=CCC1,<rdkit.Chem.rdchem.Mol object at 0x000001F1F67...,7.925579,7.925579,1.250291,-3.407176,0.452662,79.500000,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
758,99-30-9,1,Nc1c(Cl)cc([N+](=O)[O-])cc1Cl,<rdkit.Chem.rdchem.Mol object at 0x000001F1F67...,10.569109,10.569109,0.004437,-0.953930,0.436427,14.166667,...,0,0,0,0,0,0,0,0,0,0
759,99387-89-0,2,CCCOCC(=Nc1ccc(Cl)cc1C(F)(F)F)n1ccnc1,<rdkit.Chem.rdchem.Mol object at 0x000001F1F67...,13.858852,13.858852,0.050830,-5.501763,0.452489,24.521739,...,0,0,0,0,0,0,0,0,0,0
760,99-65-0,1,O=[N+]([O-])c1cccc([N+](=O)[O-])c1,<rdkit.Chem.rdchem.Mol object at 0x000001F1F67...,10.479335,10.479335,0.935741,-1.143704,0.494119,14.000000,...,0,0,0,0,0,0,0,0,0,0
761,99-71-8,3,CCC(C)c1ccc(O)cc1,<rdkit.Chem.rdchem.Mol object at 0x000001F1F67...,8.182117,8.182117,0.860105,-3.750924,0.686860,37.454545,...,0,0,0,0,0,0,0,0,0,0


In [2]:
nan_columns = df.columns[df.isna().any()].tolist()
nan_rows = df[df.isna().any(axis=1)]

print("Columns with NaN values:", nan_columns)
print("Number of rows with NaN values:", len(nan_rows))

nan_rows

Columns with NaN values: ['MaxPartialCharge', 'MinPartialCharge', 'MaxAbsPartialCharge', 'MinAbsPartialCharge', 'BCUT2D_MWHI', 'BCUT2D_MWLOW', 'BCUT2D_CHGHI', 'BCUT2D_CHGLO', 'BCUT2D_LOGPHI', 'BCUT2D_LOGPLOW', 'BCUT2D_MRHI', 'BCUT2D_MRLOW']
Number of rows with NaN values: 7


,CAS,Class,QSAR_READY_SMILES,mol,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,...,fr_priamide,fr_pyridine,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_thiazole,fr_thiocyan,fr_thiophene,fr_urea
41,1067-97-6,2,CCCC[Sn](CCCC)CCCC.O,<rdkit.Chem.rdchem.Mol object at 0x000001F1F67...,8.363501,8.363501,2.750000,-7.252575,0.575429,52.214286,...,0,0,0,0,0,0,0,0,0,0
385,41083-11-8,1,c1ncn([Sn](C2CCCCC2)(C2CCCCC2)C2CCCCC2)n1,<rdkit.Chem.rdchem.Mol object at 0x000001F1F67...,10.672299,10.087043,0.960171,-10.672299,0.536477,82.750000,...,0,0,0,0,0,0,0,0,0,0
454,56-35-9,2,CCCC[Sn](CCCC)(CCCC)O[Sn](CCCC)(CCCC)CCCC,<rdkit.Chem.rdchem.Mol object at 0x000001F1F67...,12.014988,9.512798,5.050126,-12.014988,0.127319,54.666667,...,0,0,0,0,0,0,0,0,0,0
525,629-25-4,3,CCCCCCCCCCCC(=O)[O-].[Na+],<rdkit.Chem.rdchem.Mol object at 0x000001F1F67...,11.048216,11.048216,0.000000,-4.820944,0.374681,42.800000,...,0,0,0,0,0,0,0,0,0,0
535,639-58-7,1,Cl[Sn](c1ccccc1)(c1ccccc1)c1ccccc1,<rdkit.Chem.rdchem.Mol object at 0x000001F1F67...,8.459817,8.459817,0.621270,-6.122956,0.608506,21.750000,...,0,0,0,0,0,0,0,0,0,0
601,76-87-9,2,O.c1ccc([Sn](c2ccccc2)c2ccccc2)cc1,<rdkit.Chem.rdchem.Mol object at 0x000001F1F67...,8.415720,8.415720,0.383832,-4.704311,0.626241,21.550000,...,0,0,0,0,0,0,0,0,0,0
682,900-95-8,2,CC(=O)O[Sn](c1ccccc1)(c1ccccc1)c1ccccc1,<rdkit.Chem.rdchem.Mol object at 0x000001F1F67...,13.086408,13.086408,0.964622,-7.124982,0.618505,22.608696,...,0,0,0,0,0,0,0,0,0,0


In [3]:
cols = ['CAS', 'Class', 'QSAR_READY_SMILES'] + nan_columns
nan_rows.loc[:, cols]

,CAS,Class,QSAR_READY_SMILES,MaxPartialCharge,MinPartialCharge,MaxAbsPartialCharge,MinAbsPartialCharge,BCUT2D_MWHI,BCUT2D_MWLOW,BCUT2D_CHGHI,BCUT2D_CHGLO,BCUT2D_LOGPHI,BCUT2D_LOGPLOW,BCUT2D_MRHI,BCUT2D_MRLOW
41,1067-97-6,2,CCCC[Sn](CCCC)CCCC.O,0.205755,0.205755,0.205755,0.205755,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
385,41083-11-8,1,c1ncn([Sn](C2CCCCC2)(C2CCCCC2)C2CCCCC2)n1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
454,56-35-9,2,CCCC[Sn](CCCC)(CCCC)O[Sn](CCCC)(CCCC)CCCC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
525,629-25-4,3,CCCCCCCCCCCC(=O)[O-].[Na+],1.000000,-0.550172,1.000000,0.550172,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
535,639-58-7,1,Cl[Sn](c1ccccc1)(c1ccccc1)c1ccccc1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
601,76-87-9,2,O.c1ccc([Sn](c2ccccc2)c2ccccc2)cc1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
682,900-95-8,2,CC(=O)O[Sn](c1ccccc1)(c1ccccc1)c1ccccc1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
